# Auditoría piloto de AIME

Este notebook resume la auditoría mínima de `disco-eth/AIME` como dataset candidato para el TFM.

Pregunta de decisión: AIME es técnicamente véálido como dataset principal para comparar un baseline `MFCC + SVM` frente a un enfoque con encoder profundo preentrenado en una tarea binaria humano/IA sobre fragmentos musicales?

## 0. Objetivo y criterio de aceptación

Se valida `disco-eth/AIME` para clasificación binaria de fragmentos musicales:

- `label = 0`: `model == "MTG-Jamendo"`.
- `label = 1`: resto de modelos.

La unidad experimental será el fragmento de audio. La futura aplicación no debe interpretarse como certificación de canciones completas: deberá agregar scores de varios fragmentos para producir un score global.

Estados posibles:

- `aime_apto`: dataset directamente utilizable sin condiciones relevantes detectadas.
- `aime_apto_con_condiciones`: dataset utilizable si se respetan las condiciones técnicas observadas.
- `aime_no_confirmado`: evidencia insuficiente o fallo relevante de acceso, audio o features.

Resultado observado de la auditoría: `aime_apto_con_condiciones`.

## 1. Entorno y versiones

In [1]:
import os
import re
import sys
import time
import hashlib
from pathlib import Path
from importlib import metadata

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import soundfile as sf
from huggingface_hub import HfApi, HfFileSystem

repo_root = Path.cwd() if (Path.cwd() / ".git").exists() else Path.cwd().parent
os.environ.setdefault("HF_HOME", str(repo_root / ".cache" / "huggingface"))

packages = [
    "datasets",
    "huggingface_hub",
    "pyarrow",
    "numpy",
    "pandas",
    "soundfile",
    "librosa",
    "torch",
    "transformers",
]
versions = {
    "python": sys.version.replace("\n", " "),
    "python_executable": sys.executable,
}
for package in packages:
    try:
        versions[package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        versions[package] = "NOT INSTALLED"

versions

c:\Users\sergio\tfm\tfm-ai-music-detection\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'python': '3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]',
 'python_executable': 'c:\\Users\\sergio\\tfm\\tfm-ai-music-detection\\.venv\\Scripts\\python.exe',
 'datasets': '5.0.0',
 'huggingface_hub': '1.21.0',
 'pyarrow': '24.0.0',
 'numpy': '2.4.6',
 'pandas': '3.0.3',
 'soundfile': '0.14.0',
 'librosa': '0.11.0',
 'torch': '2.13.0',
 'transformers': '5.13.1'}

## 2. Metadatos mínimos de AIME

Se leen solo las columnas `id`, `model` y `description`. La columna `audio` no se materializa en esta sección.

La caché ligera `data/interim/aime_metadata_minimal.parquet` evita repetir lecturas remotas largas. Si no existe, se reconstruye con PyArrow y `HfFileSystem` proyectando únicamente las tres columnas de metadatos.

In [2]:
dataset_id = "disco-eth/AIME"
revision = "b84d4be5eda830b6eb714998569dba73530f2601"
metadata_columns = ["id", "model", "description"]
metadata_cache = repo_root / "data" / "interim" / "aime_metadata_minimal.parquet"

api = HfApi()
hf_fs = HfFileSystem()

if metadata_cache.exists():
    metadata_df = pd.read_parquet(metadata_cache)
    metadata_source = "local_cache"
else:
    repo_files = list(
        api.list_repo_tree(
            dataset_id,
            repo_type="dataset",
            revision=revision,
            path_in_repo="data",
            recursive=True,
            expand=True,
        )
    )
    parquet_files = sorted(
        [file for file in repo_files if getattr(file, "path", "").endswith(".parquet")],
        key=lambda file: file.path,
    )
    parquet_paths = [f"datasets/{dataset_id}@{revision}/{file.path}" for file in parquet_files]
    table = pq.read_table(parquet_paths, filesystem=hf_fs, columns=metadata_columns)
    metadata_df = table.to_pandas()
    metadata_cache.parent.mkdir(parents=True, exist_ok=True)
    metadata_df.to_parquet(metadata_cache, index=False)
    metadata_source = "remote_parquet_projection"

metadata_df = metadata_df[metadata_columns].copy()
assert list(metadata_df.columns) == metadata_columns

human_mask = metadata_df["model"].eq("MTG-Jamendo")
ai_mask = ~human_mask
model_counts = metadata_df["model"].value_counts().sort_index()
description_group_sizes = metadata_df.groupby("description", dropna=False).size()
null_counts = metadata_df.isna().sum()
duplicate_ids = int(metadata_df["id"].duplicated().sum())

metadata_summary = {
    "metadata_source": metadata_source,
    "rows": int(len(metadata_df)),
    "columns": list(metadata_df.columns),
    "model_counts": model_counts.to_dict(),
    "human_rows": int(human_mask.sum()),
    "ai_rows": int(ai_mask.sum()),
    "unique_descriptions": int(metadata_df["description"].nunique(dropna=False)),
    "group_size_distribution": description_group_sizes.value_counts().sort_index().to_dict(),
    "null_counts": null_counts.to_dict(),
    "duplicate_ids": duplicate_ids,
    "subset_500_500_feasible": bool(human_mask.sum() == 500 and ai_mask.sum() >= 500 and metadata_df["description"].nunique(dropna=False) >= 500),
}

metadata_summary

{'metadata_source': 'remote_parquet_projection',
 'rows': 6500,
 'columns': ['id', 'model', 'description'],
 'model_counts': {'AudioLDM 2 Large': 500,
  'AudioLDM 2 Music': 500,
  'MTG-Jamendo': 500,
  'MusicGen Large': 500,
  'MusicGen Medium': 500,
  'MusicGen Small': 500,
  'Mustango': 500,
  'Riffusion': 500,
  'Stable Audio v1': 500,
  'Stable Audio v2': 500,
  'Suno v3': 500,
  'Suno v3.5': 500,
  'Udio': 500},
 'human_rows': 500,
 'ai_rows': 6000,
 'unique_descriptions': 500,
 'group_size_distribution': {13: 500},
 'null_counts': {'id': 0, 'model': 0, 'description': 0},
 'duplicate_ids': 0,
 'subset_500_500_feasible': True}

## 3. Validación de etiquetas y agrupación

`description` es la unidad de agrupación semántica disponible en AIME. No es un `song_id` ni identifica de forma verificable una canción original de MTG-Jamendo.

La separación futura de entrenamiento/validación/test debe hacerse por `description` para evitar fugas entre ejemplos que comparten la misma descripción.

In [3]:
metadata_df["label"] = np.where(metadata_df["model"].eq("MTG-Jamendo"), 0, 1)

label_is_consistent = bool(
    metadata_df.loc[metadata_df["model"].eq("MTG-Jamendo"), "label"].eq(0).all()
    and metadata_df.loc[~metadata_df["model"].eq("MTG-Jamendo"), "label"].eq(1).all()
)

group_audit = (
    metadata_df.groupby("description", dropna=False)
    .agg(
        rows=("id", "size"),
        human_rows=("label", lambda values: int((values == 0).sum())),
        ai_rows=("label", lambda values: int((values == 1).sum())),
        unique_models=("model", "nunique"),
    )
)

anomalous_groups = group_audit[
    ~(
        group_audit["rows"].eq(13)
        & group_audit["human_rows"].eq(1)
        & group_audit["ai_rows"].eq(12)
        & group_audit["unique_models"].eq(13)
    )
]

groups_are_regular = anomalous_groups.empty
label_grouping_summary = {
    "label_is_consistent": label_is_consistent,
    "groups_are_regular": bool(groups_are_regular),
    "groups": int(len(group_audit)),
    "anomalous_groups": int(len(anomalous_groups)),
    "group_size_distribution": group_audit["rows"].value_counts().sort_index().to_dict(),
}

label_grouping_summary

{'label_is_consistent': True,
 'groups_are_regular': True,
 'groups': 500,
 'anomalous_groups': 0,
 'group_size_distribution': {13: 500}}

## 4. Prueba mínima de audio

La prueba de aceptación usa una única `description` completa: `ambient, blues, piano`. Debe contener 13 filas: un ejemplo `MTG-Jamendo` y uno de cada generador IA.

Los audios se guardan en `data/audio/aime_acceptance_raw/`, una ruta ignorada por Git. Si faltara alguno, la celda descarga solo ese audio mediante PyArrow, leyendo la columna `audio` exclusivamente para el `id` necesario.

In [4]:
acceptance_description = "ambient, blues, piano"
acceptance_raw_dir = repo_root / "data" / "audio" / "aime_acceptance_raw"
acceptance_raw_dir.mkdir(parents=True, exist_ok=True)

acceptance_rows_df = (
    metadata_df[metadata_df["description"].eq(acceptance_description)]
    .sort_values("model")
    [["id", "model", "description", "label"]]
    .reset_index(drop=True)
)
assert len(acceptance_rows_df) == 13
assert acceptance_rows_df["model"].nunique() == 13

# Manifest verificado durante la auditoría. Permite completar la muestra sin escanear audio globalmente.
acceptance_manifest = pd.DataFrame([
    {"id": "01631", "model": "AudioLDM 2 Large", "shard": "data/train-00069-of-00210.parquet"},
    {"id": "02131", "model": "AudioLDM 2 Music", "shard": "data/train-00089-of-00210.parquet"},
    {"id": "06001", "model": "MTG-Jamendo", "shard": "data/train-00019-of-00210.parquet"},
    {"id": "01131", "model": "MusicGen Large", "shard": "data/train-00048-of-00210.parquet"},
    {"id": "00631", "model": "MusicGen Medium", "shard": "data/train-00028-of-00210.parquet"},
    {"id": "00131", "model": "MusicGen Small", "shard": "data/train-00007-of-00210.parquet"},
    {"id": "03001", "model": "Mustango", "shard": "data/train-00120-of-00210.parquet"},
    {"id": "02631", "model": "Riffusion", "shard": "data/train-00107-of-00210.parquet"},
    {"id": "03501", "model": "Stable Audio v1", "shard": "data/train-00137-of-00210.parquet"},
    {"id": "04001", "model": "Stable Audio v2", "shard": "data/train-00154-of-00210.parquet"},
    {"id": "05001", "model": "Suno v3", "shard": "data/train-00019-of-00210.parquet"},
    {"id": "05501", "model": "Suno v3.5", "shard": "data/train-00188-of-00210.parquet"},
    {"id": "04501", "model": "Udio", "shard": "data/train-00000-of-00210.parquet"},
])

acceptance_rows_df

,id,model,description,label
0,01631,AudioLDM 2 Large,"ambient, blues, piano",1
1,02131,AudioLDM 2 Music,"ambient, blues, piano",1
2,06001,MTG-Jamendo,"ambient, blues, piano",0
3,01131,MusicGen Large,"ambient, blues, piano",1
4,00631,MusicGen Medium,"ambient, blues, piano",1
5,00131,MusicGen Small,"ambient, blues, piano",1
6,03001,Mustango,"ambient, blues, piano",1
7,02631,Riffusion,"ambient, blues, piano",1
8,03501,Stable Audio v1,"ambient, blues, piano",1
9,04001,Stable Audio v2,"ambient, blues, piano",1


In [5]:
def safe_slug(value: str) -> str:
    return re.sub(r"[^a-z0-9]+", "-", value.lower()).strip("-")

def audio_filename(model: str, audio_id: str) -> str:
    return f"{safe_slug(model)}_{audio_id}.wav"

def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def download_single_audio(row: pd.Series, local_path: Path) -> None:
    parquet_path = f"datasets/{dataset_id}@{revision}/{row['shard']}"
    table = pq.read_table(
        parquet_path,
        filesystem=hf_fs,
        columns=["id", "audio"],
        filters=[("id", "=", row["id"])],
    )
    if table.num_rows != 1:
        raise ValueError(f"No se obtuvo exactamente un audio para id={row['id']}")
    audio_value = table.column("audio")[0].as_py()
    audio_bytes = audio_value.get("bytes") if isinstance(audio_value, dict) else None
    if not isinstance(audio_bytes, (bytes, bytearray)):
        raise ValueError(f"audio.bytes no disponible para id={row['id']}")
    local_path.write_bytes(audio_bytes)

raw_audio_records = []
audio_errors = []
for _, row in acceptance_manifest.iterrows():
    label = int(acceptance_rows_df.loc[acceptance_rows_df["id"].eq(row["id"]), "label"].iloc[0])
    local_path = acceptance_raw_dir / audio_filename(row["model"], row["id"])
    if not local_path.exists():
        try:
            download_single_audio(row, local_path)
        except Exception as exc:
            audio_errors.append({"id": row["id"], "model": row["model"], "error": repr(exc)})

    record = {
        "id": row["id"],
        "model": row["model"],
        "label": label,
        "local_path": str(local_path),
        "sha256": None,
        "file_size_bytes": None,
        "format": None,
        "subtype": None,
        "sample_rate": None,
        "channels": None,
        "duration_seconds": None,
        "frames": None,
        "decoded": False,
        "error": None,
    }
    if local_path.exists():
        try:
            info = sf.info(str(local_path))
            with sf.SoundFile(str(local_path)) as audio_file:
                frames = len(audio_file)
            record.update({
                "sha256": file_sha256(local_path),
                "file_size_bytes": local_path.stat().st_size,
                "format": info.format,
                "subtype": info.subtype,
                "sample_rate": info.samplerate,
                "channels": info.channels,
                "duration_seconds": info.duration,
                "frames": frames,
                "decoded": True,
            })
        except Exception as exc:
            record["error"] = repr(exc)
            audio_errors.append({"id": row["id"], "model": row["model"], "error": repr(exc)})
    raw_audio_records.append(record)

raw_audio_df = pd.DataFrame(raw_audio_records).sort_values("model").reset_index(drop=True)
raw_audio_df

,id,model,label,local_path,sha256,file_size_bytes,format,subtype,sample_rate,channels,duration_seconds,frames,decoded,error
0,01631,AudioLDM 2 Large,1,c:\Users\sergio\tfm\tfm-ai-music-detection\dat...,3b8af73b33425979eea5b7ccec01fd55c05017629ba94e...,640058,WAV,FLOAT,16000,1,10.000000,160000,True,None
1,02131,AudioLDM 2 Music,1,c:\Users\sergio\tfm\tfm-ai-music-detection\dat...,1c05a93f5e1a1d79ecd58dd531bf4969ecbabf9798e349...,640058,WAV,FLOAT,16000,1,10.000000,160000,True,None
2,06001,MTG-Jamendo,0,c:\Users\sergio\tfm\tfm-ai-music-detection\dat...,e7fa28f19eba20d67e95d5c15b0abfd6b9fe85fef9866d...,49177158,WAV,PCM_16,48000,2,256.130625,12294270,True,None
3,01131,MusicGen Large,1,c:\Users\sergio\tfm\tfm-ai-music-detection\dat...,688af020ecac16f9e7c2f16a0a548b8ad966347fbcbe26...,1303098,WAV,FLOAT,32000,1,10.180000,325760,True,None
4,00631,MusicGen Medium,1,c:\Users\sergio\tfm\tfm-ai-music-detection\dat...,0b27b0d35495f2497b6287f75c009f4f9726f261531358...,1303098,WAV,FLOAT,32000,1,10.180000,325760,True,None
5,00131,MusicGen Small,1,c:\Users\sergio\tfm\tfm-ai-music-detection\dat...,a94d245ad4b954a148819f4d2e7e7c1ce53a56efddf87c...,1303098,WAV,FLOAT,32000,1,10.180000,325760,True,None
6,03001,Mustango,1,c:\Users\sergio\tfm\tfm-ai-music-detection\dat...,a531b12376fade7e1362402e24cdc8d8711a6ee199e2cd...,327788,WAV,PCM_16,16000,1,10.242000,163872,True,None
7,02631,Riffusion,1,c:\Users\sergio\tfm\tfm-ai-music-detection\dat...,abe7fc43d88e94543412734b842e73cff89b4b66861d70...,902364,WAV,PCM_16,44100,1,10.230000,451143,True,None
8,03501,Stable Audio v1,1,c:\Users\sergio\tfm\tfm-ai-music-detection\dat...,610058d22f79181b14347aaee54dcfcfab4d0ea488694b...,1920078,WAV,PCM_16,48000,2,10.000000,480000,True,None
9,04001,Stable Audio v2,1,c:\Users\sergio\tfm\tfm-ai-music-detection\dat...,aef446f6907bc2f569eea7dbd3d99d8b039887e934d140...,1920078,WAV,PCM_16,48000,2,10.000000,480000,True,None


## 5. Preprocesamiento común

Se valida un pipeline común mínimo: recorte de 10 segundos, conversión a mono y resample a 16 kHz.

Condiciones importantes:

- `16 kHz` es una decisión provisional de auditoría, no una decisión final del experimento profundo.
- El sample rate definitivo podrá depender del encoder seleccionado.
- El recorte por máxima energía media es un pipeline práctico del TFM; no pretende reproducir exactamente el algoritmo interno de AIME.

In [6]:
import librosa

def extract_10s_mono_resampled(
    path: str | Path,
    target_duration_seconds: float = 10.0,
    target_sample_rate: int = 16000,
    energy_hop_seconds: float = 0.5,
) -> dict:
    try:
        audio, source_sample_rate = sf.read(str(path), always_2d=True, dtype="float32")
        mono = audio.mean(axis=1).astype(np.float32)
        source_duration = len(mono) / source_sample_rate
        target_source_samples = int(round(target_duration_seconds * source_sample_rate))

        if source_duration + 1e-6 < target_duration_seconds:
            return {"waveform": None, "sample_rate": target_sample_rate, "start_seconds": None, "error": "audio inferior a 10 segundos"}

        if len(mono) > target_source_samples:
            hop_samples = max(1, int(round(energy_hop_seconds * source_sample_rate)))
            starts = np.arange(0, len(mono) - target_source_samples + 1, hop_samples)
            if starts[-1] != len(mono) - target_source_samples:
                starts = np.append(starts, len(mono) - target_source_samples)
            energies = np.array([
                np.mean(np.square(mono[start:start + target_source_samples]))
                for start in starts
            ])
            start_sample = int(starts[int(np.argmax(energies))])
        else:
            start_sample = 0

        segment = mono[start_sample:start_sample + target_source_samples]
        if source_sample_rate != target_sample_rate:
            segment = librosa.resample(segment, orig_sr=source_sample_rate, target_sr=target_sample_rate).astype(np.float32)

        target_samples = int(round(target_duration_seconds * target_sample_rate))
        delta = len(segment) - target_samples
        if abs(delta) <= 2:
            segment = segment[:target_samples] if delta > 0 else np.pad(segment, (0, -delta))
        if len(segment) != target_samples:
            return {"waveform": None, "sample_rate": target_sample_rate, "start_seconds": start_sample / source_sample_rate, "error": f"longitud inesperada: {len(segment)}"}
        if not np.all(np.isfinite(segment)):
            return {"waveform": None, "sample_rate": target_sample_rate, "start_seconds": start_sample / source_sample_rate, "error": "valores no finitos"}

        return {"waveform": segment.astype(np.float32), "sample_rate": target_sample_rate, "start_seconds": start_sample / source_sample_rate, "error": None}
    except Exception as exc:
        return {"waveform": None, "sample_rate": target_sample_rate, "start_seconds": None, "error": repr(exc)}

target_sample_rate = 16000
target_num_samples = 160000
preprocess_records = []
waveforms = []
for row in raw_audio_df.itertuples(index=False):
    result = extract_10s_mono_resampled(row.local_path, target_sample_rate=target_sample_rate)
    waveform = result["waveform"]
    valid = waveform is not None and waveform.ndim == 1 and len(waveform) == target_num_samples and np.all(np.isfinite(waveform))
    if valid:
        waveforms.append(waveform)
    preprocess_records.append({
        "id": row.id,
        "model": row.model,
        "start_seconds": result["start_seconds"],
        "num_samples": len(waveform) if waveform is not None else None,
        "valid": bool(valid),
        "error": result["error"],
    })

preprocess_df = pd.DataFrame(preprocess_records)
preprocess_errors = preprocess_df[preprocess_df["error"].notna()].to_dict(orient="records")
waveforms = np.stack(waveforms).astype(np.float32) if len(waveforms) == 13 else np.empty((0, target_num_samples), dtype=np.float32)
preprocess_df

,id,model,start_seconds,num_samples,valid,error
0,01631,AudioLDM 2 Large,0.00,160000,True,None
1,02131,AudioLDM 2 Music,0.00,160000,True,None
2,06001,MTG-Jamendo,78.00,160000,True,None
3,01131,MusicGen Large,0.00,160000,True,None
4,00631,MusicGen Medium,0.00,160000,True,None
5,00131,MusicGen Small,0.18,160000,True,None
6,03001,Mustango,0.00,160000,True,None
7,02631,Riffusion,0.00,160000,True,None
8,03501,Stable Audio v1,0.00,160000,True,None
9,04001,Stable Audio v2,0.00,160000,True,None


## 6. Extracción MFCC

No se entrena ningún clasificador. Solo se valida que los fragmentos preprocesados permiten obtener features MFCC finitas.

In [7]:
mfcc_features = []
for waveform in waveforms:
    mfcc = librosa.feature.mfcc(y=waveform, sr=target_sample_rate, n_mfcc=20)
    mfcc_features.append(np.concatenate([mfcc.mean(axis=1), mfcc.std(axis=1)]).astype(np.float32))

X_mfcc = np.stack(mfcc_features) if mfcc_features else np.empty((0, 40), dtype=np.float32)
mfcc_all_finite = bool(np.all(np.isfinite(X_mfcc)))

mfcc_summary = {
    "mfcc_shape": X_mfcc.shape,
    "mfcc_all_finite": mfcc_all_finite,
}
mfcc_summary

{'mfcc_shape': (13, 40), 'mfcc_all_finite': True}

## 7. Smoke test de embeddings

El modelo `MIT/ast-finetuned-audioset-10-10-0.4593` se usó únicamente como prueba técnica de compatibilidad. No queda seleccionado como encoder definitivo del TFM.

No se entrenó, no se hizo fine-tuning y no se guardaron embeddings en disco.

Para mantener el notebook ligero, la celda de ejecución del encoder queda desactivada por defecto. El resultado observado de la auditoría fue:

- `X_embed.shape == (13, 768)`.
- Todos los valores finitos.
- Dispositivo: CPU.
- Modelo usado solo como smoke test: `MIT/ast-finetuned-audioset-10-10-0.4593`.

In [8]:
RUN_EMBEDDING_SMOKE_TEST = False
embedding_model_id = "MIT/ast-finetuned-audioset-10-10-0.4593"
embedding_device = "cpu"
embedding_shape = (13, 768)
embedding_all_finite = True
embedding_error = None
X_embed = None

if RUN_EMBEDDING_SMOKE_TEST:
    try:
        import torch
        from transformers import AutoModel, AutoProcessor

        embedding_device = "cuda" if torch.cuda.is_available() else "cpu"
        processor = AutoProcessor.from_pretrained(embedding_model_id)
        model = AutoModel.from_pretrained(embedding_model_id).to(embedding_device)
        model.eval()
        embeddings = []
        with torch.no_grad():
            for waveform in waveforms:
                inputs = processor(waveform, sampling_rate=target_sample_rate, return_tensors="pt")
                inputs = {key: value.to(embedding_device) for key, value in inputs.items()}
                outputs = model(**inputs)
                embedding = outputs.last_hidden_state.mean(dim=1).squeeze(0)
                embeddings.append(embedding.detach().cpu().numpy().astype(np.float32))
        X_embed = np.stack(embeddings)
        embedding_shape = X_embed.shape
        embedding_all_finite = bool(np.all(np.isfinite(X_embed)))
    except Exception as exc:
        embedding_shape = None
        embedding_all_finite = False
        embedding_error = repr(exc)

embedding_summary = {
    "embedding_model_id": embedding_model_id,
    "embedding_shape": embedding_shape,
    "embedding_all_finite": embedding_all_finite,
    "embedding_device": embedding_device,
    "embedding_error": embedding_error,
    "executed_now": RUN_EMBEDDING_SMOKE_TEST,
}
embedding_summary

{'embedding_model_id': 'MIT/ast-finetuned-audioset-10-10-0.4593',
 'embedding_shape': (13, 768),
 'embedding_all_finite': True,
 'embedding_device': 'cpu',
 'embedding_error': None,
 'executed_now': False}

## 8. Veredicto técnico

Condiciones del veredicto:

- AIME trabaja con fragmentos, no con canciones completas.
- La clase humana disponible es `MTG-Jamendo`.
- La clase IA agrupa 12 generadores distintos.
- Los audios brutos son heterogéneos en duración, sample rate, canales y subtipo.
- El experimento debe aplicar un preprocesamiento común antes de MFCC o embeddings.
- `description` es el grupo semántico disponible para particionar sin fugas; no es un identificador de canción.
- La aplicación futura deberá comunicar scores por fragmento y score agregado, no certificación absoluta.

In [9]:
decoded_audio_files = int(raw_audio_df["decoded"].sum()) if "raw_audio_df" in globals() else 0
preprocessing_ok = bool(len(waveforms) == 13 and waveforms.shape == (13, target_num_samples)) if "waveforms" in globals() else False
mfcc_ok = bool(X_mfcc.shape == (13, 40) and mfcc_all_finite) if "X_mfcc" in globals() else False
embedding_ok = bool(embedding_shape == (13, 768) and embedding_all_finite)
subset_500_500_feasible = bool(metadata_summary["subset_500_500_feasible"])

audio_errors = audio_errors if "audio_errors" in globals() else []

if (
    metadata_summary["rows"] == 6500
    and metadata_summary["human_rows"] == 500
    and metadata_summary["ai_rows"] == 6000
    and label_is_consistent
    and groups_are_regular
    and subset_500_500_feasible
    and decoded_audio_files == 13
    and preprocessing_ok
    and mfcc_ok
    and embedding_ok
):
    acceptance_status = "aime_apto_con_condiciones"
else:
    acceptance_status = "aime_no_confirmado"

final_summary = {
    "metadata_rows": metadata_summary["rows"],
    "human_rows": metadata_summary["human_rows"],
    "ai_rows": metadata_summary["ai_rows"],
    "unique_models": int(metadata_df["model"].nunique()),
    "unique_descriptions": metadata_summary["unique_descriptions"],
    "groups_are_regular": bool(groups_are_regular),
    "subset_500_500_feasible": subset_500_500_feasible,
    "selected_audio_rows": int(len(acceptance_rows_df)) if "acceptance_rows_df" in globals() else 0,
    "decoded_audio_files": decoded_audio_files,
    "preprocessing_ok": preprocessing_ok,
    "mfcc_shape": X_mfcc.shape if "X_mfcc" in globals() else None,
    "mfcc_all_finite": mfcc_all_finite if "mfcc_all_finite" in globals() else None,
    "embedding_shape": embedding_shape,
    "embedding_all_finite": embedding_all_finite,
    "audio_errors": audio_errors,
    "acceptance_status": acceptance_status,
}

print("VEREDICTO TÉCNICO AIME")
for key, value in final_summary.items():
    print(f"{key}: {value}")

VEREDICTO TÉCNICO AIME
metadata_rows: 6500
human_rows: 500
ai_rows: 6000
unique_models: 13
unique_descriptions: 500
groups_are_regular: True
subset_500_500_feasible: True
selected_audio_rows: 13
decoded_audio_files: 13
preprocessing_ok: True
mfcc_shape: (13, 40)
mfcc_all_finite: True
embedding_shape: (13, 768)
embedding_all_finite: True
audio_errors: []
acceptance_status: aime_apto_con_condiciones
